In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from datetime import datetime
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
import re

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_columns', None)
NOME_ARQUIVO = 'basetreinamento.xlsx'
print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


# 🔄 Pipeline de Detecção de FPD (First Payment Default)

## Fluxo do projeto:
1. **Carregamento e Limpeza** - Remover colunas irrelevantes
2. **Tratamento de Dados** - Processar categorias e datas
3. **Preparação para Modelagem** - Separar X/y, train/test split, balanceamento
4. **Visualizações** - Explorar distribuições e padrões
5. **Modelagem** - Treinar modelos
6. **Avaliação** - Gerar métricas e resultados

In [16]:
df = pd.read_excel(NOME_ARQUIVO, nrows=5000)
df.head()

,pedido_id,CPF,SCORE_HCP4,SCORE_HCP5,SCORE_HEST,SCORE_HFI4,SCORE_HFI5,HI01_PROB,HI01_CONCEITO,SCORE_HIPA,SCORE_HIPN,HPG5,HCR5,H5OR,SCORE_HRM5,SCORE_HSV4,SCORE_HSV5,SCORE_HVA4,SCORE_HVA5,MENSAGEM_TIPO_REGISTRO,FPD,produtor,lancamento,segmento,status_cobranca,status_financeiro,status_pedido,nome,email,score,categoria_risco_score,documento,endereco_cep,endereco_estado,endereco_cidade,nascimento,idade,telefone_ativo,modalidade,total_financiado,quantidade_parcelas,saldo_vencido,quantidade_parcelas_vencidas,recebido,primeiro_vencimento_em_atraso,dias_em_atraso,pdd,saldo_vencido_com_juros,total_pago_com_juros,aguardando_pagamento_sem_juros,vencidos_sem_juros_tmb,recebido_sem_juros_tmb,data_efetivacao,data_quitacao,order_bump,pedido_pai_ob
0,26414,D6CEA19DCA,567,567,1,558,570,NaN,NaN,99,100,NaN,NaN,NaN,NaN,500,538,458,494,APTOS,NÃ£o,Produtor_1583,Lancamento_2563,Autoconhecimento,CobranÃ§a Interna,Inadimplente,Efetivado,Cliente_25AAFE70,user_6DDD6F8B@anon.local,130,Alto,D6CEA19DCA,88220000,Santa Catarina,Cidade_1893,1978-12-12 00:00:00,47,BDECE65488,Compartilhado,1944.22,13.0,743.70,5.0,1200.52,2024-11-20 00:00:00,434,False,743.70,1244.15,NaN,533.25,932.35,2024-08-21 16:48:00,NaT,NaT,NaN
1,60182,BC8B4630C6,580,567,6,679,621,0.099992,0.0,99,94,191.0,88.0,50698.0,1599.0,575,623,673,598,APTOS,NÃ£o,Produtor_0238,Lancamento_2456,Empreendedor,NaN,Adimplente,Cancelamento Solicitado,Cliente_45B91102,user_9612B4E0@anon.local,196,Alto,BC8B4630C6,39540000,MG,Cidade_3985,1981-09-14 00:00:00,44,6F2C9BC475,Compartilhado,129.7,1.0,NaN,0.0,129.70,NaN,NaN,False,NaN,129.70,NaN,NaN,97.28,2024-06-07 00:12:00,NaT,NaT,NaN
2,938745,BC8B4630C6,580,567,6,679,621,0.099992,0.0,99,94,191.0,88.0,50698.0,1599.0,575,623,673,598,APTOS,NÃ£o,Produtor_0181,Lancamento_2660,Empreendedor,NaN,Quitado,Efetivado,Cliente_CF1B01F0,user_9612B4E0@anon.local,710,Baixo,BC8B4630C6,39540-000,MG,Cidade_3985,1981-09-14 00:00:00,44,56C8E4B75C,Antecipado,1796.04,12.0,NaN,0.0,1796.04,NaN,NaN,False,NaN,1820.73,NaN,NaN,1497.00,2024-06-07 21:59:00,2025-07-17 00:15:00,NaT,NaN
3,90049,4BA2379170,192,515,1,329,501,NaN,NaN,99,35,NaN,NaN,NaN,NaN,325,532,303,546,APTOS,NÃ£o,Produtor_0078,Lancamento_2766,Empreendedor,Negativado,Inadimplente,Efetivado,Cliente_FCB173CF,user_7979D663@anon.local,87,Alto,4BA2379170,69065-110,AM,Cidade_2348,1983-04-20 00:00:00,42,6CC30182E8,CrÃ©dito AcessÃ­vel,2216.04,12.0,1846.70,10.0,369.34,2024-09-20 00:00:00,495,False,1846.70,377.09,NaN,1539.20,307.84,2024-07-20 09:38:00,NaT,NaT,NaN
4,90063,4AA40DD2FA,399,525,4,295,463,NaN,NaN,99,0,NaN,NaN,NaN,NaN,409,523,352,501,APTOS,NÃ£o,Produtor_0078,Lancamento_2766,Empreendedor,Protesto,Inadimplente,Efetivado,Cliente_BE2D1B3A,user_60D2BF20@anon.local,0,Alto,4AA40DD2FA,75920000,GO,Cidade_3484,1988-03-18 00:00:00,37,954B9BAF90,CrÃ©dito AcessÃ­vel,2216.04,12.0,923.35,5.0,1108.02,2025-09-20 00:00:00,130,False,923.35,1130.41,153.92,769.60,923.52,2025-01-23 20:41:00,NaT,NaT,NaN


## 1️⃣ CARREGAMENTO E LIMPEZA DOS DADOS

In [17]:
to_remove = [
    'pedido_id', 'CPF', 'documento', 'documento2', 'nome', 
    'email', 'telefone_ativo', 'endereco_cep', 'endereco_cidade', 'nascimento',
    'dias_em_atraso', 'saldo_vencido', 'quantidade_parcelas_vencidas','data_quitacao', 'total_pago_com_juros','recebido', 'recebido_sem_juros_tmb',
    'saldo_vencido_com_juros','endereco_estado',
    'order_bump', 'pedido_pai_ob', 'aguardando_pagamento_sem_juros', 'vencidos_sem_juros_tmb', 'status_cobranca','status_pagamento', 'status_financeiro', 'status_pedido', 'primeiro_vencimento_em_atraso',
    'HI01_PROB', 'HI01_CONCEITO', 'HCR5', 'H5OR', 'HPG5', 'SCORE_HRM5', 'data_quitacao', 'pdd', 'segmento'
]

real_remove_columns = [col for col in to_remove if col in df.columns]

df_clean = df.drop(columns=real_remove_columns)

print(f"🗑️ Colunas removidas: {real_remove_columns}")
print(f"📉 Novas dimensões do dataset: {df_clean.shape[1]} colunas restantes.")

🗑️ Colunas removidas: ['pedido_id', 'CPF', 'documento', 'nome', 'email', 'telefone_ativo', 'endereco_cep', 'endereco_cidade', 'nascimento', 'dias_em_atraso', 'saldo_vencido', 'quantidade_parcelas_vencidas', 'data_quitacao', 'total_pago_com_juros', 'recebido', 'recebido_sem_juros_tmb', 'saldo_vencido_com_juros', 'endereco_estado', 'order_bump', 'pedido_pai_ob', 'aguardando_pagamento_sem_juros', 'vencidos_sem_juros_tmb', 'status_cobranca', 'status_financeiro', 'status_pedido', 'primeiro_vencimento_em_atraso', 'HI01_PROB', 'HI01_CONCEITO', 'HCR5', 'H5OR', 'HPG5', 'SCORE_HRM5', 'data_quitacao', 'pdd', 'segmento']
📉 Novas dimensões do dataset: 22 colunas restantes.


In [18]:
# Substitua 'df' pelo nome do seu DataFrame (ex: X_train)
print(df_clean.isna().sum())

SCORE_HCP4                0
SCORE_HCP5                0
SCORE_HEST                0
SCORE_HFI4                0
SCORE_HFI5                0
SCORE_HIPA                0
SCORE_HIPN                0
SCORE_HSV4                0
SCORE_HSV5                0
SCORE_HVA4                0
SCORE_HVA5                0
MENSAGEM_TIPO_REGISTRO    0
FPD                       0
produtor                  0
lancamento                0
score                     0
categoria_risco_score     0
idade                     0
modalidade                0
total_financiado          0
quantidade_parcelas       0
data_efetivacao           0
dtype: int64


## 2️⃣ TRATAMENTO DE VARIÁVEIS

### Processar target (FPD) e variáveis categóricas

In [19]:

print(df_clean['categoria_risco_score'].info())

<class 'pandas.Series'>
RangeIndex: 5000 entries, 0 to 4999
Series name: categoria_risco_score
Non-Null Count  Dtype 
--------------  ----- 
5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB
None


In [20]:

df_clean['FPD'] = df['FPD'].map({'Sim': 1, 'NÃ£o': 0})
df_clean['score'] = df_clean['categoria_risco_score'].fillna(301)

col = df['categoria_risco_score']

texto = col.astype(str).str.strip().str.lower()
map_texto = {'alto': 'Alto', 'mÃ©dio': 'MÃ©dio', 'medio': 'MÃ©dio', 'baixo': 'Baixo'}
cat_texto = texto.map(map_texto)

score = pd.to_numeric(col, errors='coerce')
cat_score = pd.cut(
    score,
    bins=[-np.inf, 300, 700, np.inf],
    labels=['Alto', 'MÃ©dio', 'Baixo'],
    right=True,
    include_lowest=True,
    ordered=True,
    )

print(f" Categoria de risco: {cat_score.name} e {cat_score}")


df_clean['categoria_risco_score'] = cat_texto.fillna(cat_score)
df_clean['produtor'] = df['produtor'].astype('category')
df_clean['lancamento'] = df['lancamento'].astype('category')
hoje = datetime.today()

def corrigir_idade(valor):

    if isinstance(valor, (int, float)):
        return valor

    elif isinstance(valor, (datetime, pd.Timestamp)):
        return (hoje - valor).days // 365

    return pd.NA

df_clean['idade'] = df_clean['idade'].apply(corrigir_idade)

 Categoria de risco: categoria_risco_score e 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
       ... 
4995    NaN
4996    NaN
4997    NaN
4998    NaN
4999    NaN
Name: categoria_risco_score, Length: 5000, dtype: category
Categories (3, str): ['Alto' < 'MÃ©dio' < 'Baixo']


In [21]:
y = df_clean['FPD']
X = df_clean.drop(columns=['FPD', 'categoria_risco_score', 'MENSAGEM_TIPO_REGISTRO'])
print(y.value_counts())

FPD
0    4203
1     797
Name: count, dtype: int64


In [22]:

# Divisão treino/teste
X = pd.get_dummies(X, drop_first=True)
# DIVISÃO DE DADOS
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [23]:
# This will output the total number of missing values
# Substitua 'df' pelo nome do seu DataFrame (ex: X_train)
print(X.isna().sum())

SCORE_HCP4                             0
SCORE_HCP5                             0
SCORE_HEST                             0
SCORE_HFI4                             0
SCORE_HFI5                             0
                                      ..
data_efetivacao_2023-08-08 16:25:00    0
data_efetivacao_2023-08-08 17:10:00    0
data_efetivacao_2023-10-12 10:56:00    0
data_efetivacao_2023-08-09 16:58:00    0
data_efetivacao_2023-11-28 19:25:00    0
Length: 6522, dtype: int64


In [24]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)



In [26]:
X_train_balanced.columns = [re.sub(r'[^\w]', '_', col) for col in X_train_balanced.columns]

models = [
    XGBClassifier(random_state=42),
    LGBMClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    ExtraTreesClassifier(random_state=42),
    
]

# 3. LOOP DE TREINAMENTO
for model in models:
    model.fit(X_train_balanced, y_train_balanced)
    print(f"✅ {model.__class__.__name__} treinado com sucesso.")

✅ XGBClassifier treinado com sucesso.
[LightGBM] [Info] Number of positive: 3362, number of negative: 3362
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010002 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2882
[LightGBM] [Info] Number of data points in the train set: 6724, number of used features: 220
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
✅ LGBMClassifier treinado com sucesso.
✅ RandomForestClassifier treinado com sucesso.
✅ ExtraTreesClassifier treinado com sucesso.


In [ ]:
for model in models:
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não FPD', 'FPD'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(f'Matriz de Confusão — {model.__class__.__name__}')
    plt.tight_layout()
    plt.show()

In [ ]:
## 5️⃣ MODELAGEM

### Treinar modelo(s)

In [ ]:
## 6️⃣ AVALIAÇÃO E RESULTADOS

### Testar e avaliar modelo

In [ ]:
### Gerar relatório final e exportar resultados